**1. Importação das bibliotecas**

Nesta etapa, importamos as bibliotecas necessárias para preparar os dados, realizar a validação temporal e executar a busca exaustiva de hiperparâmetros.

Será utilizada a `RandomForestRegressor`, que apresentou o melhor desempenho entre os modelos iniciais do encontro anterior, juntamente com `GridSearchCV` e `TimeSeriesSplit`.


In [2]:
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

**2. Carregamento e preparação dos dados**

Nesta etapa, carregamos o dataset e organizamos os dados cronologicamente. Em seguida, criamos os mesmos atributos temporais utilizados no encontro anterior (`lag_1`, `lag_7` e `mm_7`), garantindo que a comparação com o modelo da semana passada seja justa.


In [3]:
df = pd.read_csv("/content/mecaniqa_dataset.csv")

df["Data"] = pd.to_datetime(df["Data"])
df = df.sort_values("Data").reset_index(drop=True)

df["lag_1"] = df["Trocas_Oleo"].shift(1)
df["lag_7"] = df["Trocas_Oleo"].shift(7)
df["mm_7"] = df["Trocas_Oleo"].shift(1).rolling(7).mean()

df = df.dropna().reset_index(drop=True)

X = df[["lag_1", "lag_7", "mm_7"]]
y = df["Trocas_Oleo"]

print("Período:", df["Data"].min().date(), "até", df["Data"].max().date())
print("Quantidade de registros:", len(df))
print("Atributos utilizados:", list(X.columns))

Período: 2024-01-08 até 2025-12-31
Quantidade de registros: 708
Atributos utilizados: ['lag_1', 'lag_7', 'mm_7']


**3. Divisão cronológica entre treino e teste**

Nesta etapa, dividimos os dados respeitando a ordem temporal. Os registros mais antigos serão utilizados para treinamento e os registros mais recentes serão reservados para teste.

Essa divisão evita o uso de informações futuras no treinamento e mantém a mesma estratégia utilizada no encontro anterior, permitindo comparar o modelo otimizado com a Random Forest inicial de forma justa.


In [4]:
# Divisão cronológica: 80% para treino e 20% para teste
tamanho_treino = int(len(df) * 0.8)

X_treino = X.iloc[:tamanho_treino]
X_teste = X.iloc[tamanho_treino:]

y_treino = y.iloc[:tamanho_treino]
y_teste = y.iloc[tamanho_treino:]

print("Treino:", df["Data"].iloc[:tamanho_treino].min().date(),
      "->", df["Data"].iloc[:tamanho_treino].max().date(),
      f"({len(X_treino)} registros)")

print("Teste:", df["Data"].iloc[tamanho_treino:].min().date(),
      "->", df["Data"].iloc[tamanho_treino:].max().date(),
      f"({len(X_teste)} registros)")

Treino: 2024-01-08 -> 2025-08-11 (566 registros)
Teste: 2025-08-12 -> 2025-12-31 (142 registros)


**4. Validação temporal com TimeSeriesSplit**

Nesta etapa, configuramos o `TimeSeriesSplit` com 5 divisões temporais. Em cada divisão, o modelo será treinado utilizando dados mais antigos e validado em dados posteriores.

Essa estratégia mantém a ordem cronológica e evita o uso de informações futuras durante a seleção dos melhores hiperparâmetros.


In [5]:
tscv = TimeSeriesSplit(n_splits=5)

print("Número de divisões:", tscv.n_splits)

Número de divisões: 5


**5. Definição do grid de hiperparâmetros**

Nesta etapa, definimos as combinações de hiperparâmetros que serão testadas pelo `GridSearchCV` na `RandomForestRegressor`.

Serão avaliados diferentes níveis de quantidade de árvores, profundidade e critérios mínimos para divisão e criação das folhas. O objetivo é encontrar uma configuração que reduza o erro sem aumentar desnecessariamente a complexidade do modelo.


In [6]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

print("Hiperparâmetros que serão testados:")

for parametro, valores in param_grid.items():
    print(f"{parametro}: {valores}")

Hiperparâmetros que serão testados:
n_estimators: [50, 100, 200]
max_depth: [5, 10, 15, None]
min_samples_split: [2, 5, 10]
min_samples_leaf: [1, 2, 4]


**6. Execução do Grid Search**

Nesta etapa, utilizamos o `GridSearchCV` para testar sistematicamente todas as combinações de hiperparâmetros definidas anteriormente.

A validação será realizada com o `TimeSeriesSplit`, utilizando apenas os dados de treinamento. O critério escolhido para selecionar a melhor configuração será o **RMSE**, buscando minimizar o erro de previsão.


In [7]:
modelo_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=modelo_rf,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_treino, y_treino)

print("Melhores hiperparâmetros:")
print(grid_search.best_params_)

print(f"Melhor RMSE na validação: {-grid_search.best_score_:.4f}")

Melhores hiperparâmetros:
{'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 50}
Melhor RMSE na validação: 5.7186


**7. Avaliação do modelo otimizado**

Nesta etapa, avaliamos o melhor modelo encontrado pelo `GridSearchCV` no conjunto de teste separado anteriormente.

O conjunto de teste não foi utilizado durante a busca dos hiperparâmetros. Dessa forma, podemos medir o desempenho do modelo otimizado em dados que não participaram do treinamento nem da seleção da configuração.


In [11]:
modelo_otimizado = grid_search.best_estimator_

previsoes_otimizadas = modelo_otimizado.predict(X_teste)

mse_otimizado = mean_squared_error(
    y_teste,
    previsoes_otimizadas
)

rmse_otimizado = mse_otimizado ** 0.5

mape_otimizado = mean_absolute_percentage_error(
    y_teste,
    previsoes_otimizadas
) * 100

print(f"RMSE do modelo otimizado: {rmse_otimizado:.4f}")
print(f"MAPE do modelo otimizado: {mape_otimizado:.2f}%")

RMSE do modelo otimizado: 4.0137
MAPE do modelo otimizado: 14.35%


**8. Comparação com a Random Forest inicial**

Nesta etapa, comparamos o modelo otimizado pelo `GridSearchCV` com a `RandomForestRegressor` utilizada na semana anterior com os hiperparâmetros padrão.

A redução percentual do erro será calculada para o RMSE e para o MAPE. Valores positivos indicam que o modelo otimizado apresentou menor erro.


In [12]:
rmse_anterior = 4.607770
mape_anterior = 14.738961

reducao_rmse = ((rmse_anterior - rmse_otimizado) / rmse_anterior) * 100
reducao_mape = ((mape_anterior - mape_otimizado) / mape_anterior) * 100

print("Comparação com a Random Forest da semana passada:")
print(f"RMSE anterior: {rmse_anterior:.4f}")
print(f"RMSE otimizado: {rmse_otimizado:.4f}")
print(f"Redução do RMSE: {reducao_rmse:.2f}%")

print()

print(f"MAPE anterior: {mape_anterior:.2f}%")
print(f"MAPE otimizado: {mape_otimizado:.2f}%")
print(f"Redução do MAPE: {reducao_mape:.2f}%")

Comparação com a Random Forest da semana passada:
RMSE anterior: 4.6078
RMSE otimizado: 4.0137
Redução do RMSE: 12.89%

MAPE anterior: 14.74%
MAPE otimizado: 14.35%
Redução do MAPE: 2.64%


**9. Conclusão**

A aplicação do `GridSearchCV` com `TimeSeriesSplit(n_splits=5)` permitiu testar diferentes combinações de hiperparâmetros da `RandomForestRegressor`, mantendo a ordem temporal dos dados durante a validação.

A melhor configuração encontrada foi:

* `n_estimators = 50`
* `max_depth = 10`
* `min_samples_split = 10`
* `min_samples_leaf = 4`

No conjunto de teste, o modelo otimizado apresentou **RMSE de 4,0137 trocas/dia** e **MAPE de 14,35%**. Em comparação com a Random Forest inicial, houve uma **redução de 12,89% no RMSE** e de **2,64% no MAPE**.

Os resultados indicam que a otimização dos hiperparâmetros melhorou o desempenho da Random Forest, principalmente na redução do RMSE.
